# MetaMIRAGE Cross-Node Connectivity Test

Use this notebook from a **separate worker/Jupyter node** to verify that it can reach the shared service node running:

- **Qdrant** on port `6333`
- **Preload Coordinator** on port `8001`

This notebook is **non-destructive**. It does not create, delete, reset, restore, or modify any Qdrant collections.


In [10]:
import sys
import subprocess
from pathlib import Path

pkg_dir = Path.home() / "metamirage_jupyter_pkgs"

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--target", str(pkg_dir),
    "--no-deps",
    "qdrant-client==1.18.0",
    "portalocker==3.2.0",
])

sys.path.insert(0, str(pkg_dir))

print("Installed qdrant-client + portalocker only")

  Using cached qdrant_client-1.18.0-py3-none-any.whl.metadata (11 kB)
  Using cached portalocker-3.2.0-py3-none-any.whl.metadata (8.7 kB)
Using cached qdrant_client-1.18.0-py3-none-any.whl (398 kB)
Using cached portalocker-3.2.0-py3-none-any.whl (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [portalocker]t]
Installed qdrant-client + portalocker only


## 1. Configuration

Change only `SERVICE_NODE` to the hostname or IP address of the node where Qdrant + the coordinator are running.

Example:

```python
SERVICE_NODE = "cn123.delta.internal.ncsa.edu"
```


In [12]:
# ============================================================
# EDIT THIS
# ============================================================

SERVICE_NODE = "gpub053.delta.ncsa.illinois.edu"

QDRANT_PORT = 6333
COORDINATOR_PORT = 8001

QDRANT_URL = f"http://{SERVICE_NODE}:{QDRANT_PORT}"
COORDINATOR_URL = f"http://{SERVICE_NODE}:{COORDINATOR_PORT}"

print("Qdrant URL     :", QDRANT_URL)
print("Coordinator URL:", COORDINATOR_URL)


Qdrant URL     : http://gpub053.delta.ncsa.illinois.edu:6333
Coordinator URL: http://gpub053.delta.ncsa.illinois.edu:8001


## 2. Show this worker node's identity

Useful for confirming which node the notebook itself is running on.


In [13]:
import socket
import platform

worker_hostname = socket.gethostname()
worker_fqdn = socket.getfqdn()

print("Worker hostname:", worker_hostname)
print("Worker FQDN    :", worker_fqdn)
print("Platform       :", platform.platform())


Worker hostname: gpua080.delta.ncsa.illinois.edu
Worker FQDN    : gpua080.delta.ncsa.illinois.edu
Platform       : Linux-5.14.0-570.131.1.el9_6.x86_64-x86_64-with-glibc2.34


## 3. DNS / hostname resolution test

This checks whether the worker node can resolve the service-node hostname.


In [14]:
import socket

assert SERVICE_NODE != "CHANGE_ME", "Set SERVICE_NODE in the configuration cell first."

try:
    resolved_ip = socket.gethostbyname(SERVICE_NODE)
    print(f"✅ DNS resolution successful: {SERVICE_NODE} -> {resolved_ip}")
except Exception as e:
    print(f"❌ DNS resolution failed for {SERVICE_NODE}")
    print(type(e).__name__, str(e))
    raise


✅ DNS resolution successful: gpub053.delta.ncsa.illinois.edu -> 141.142.254.153


## 4. Raw TCP port test

This verifies basic node-to-node connectivity before testing any application protocol.


In [15]:
import socket

def test_tcp(host, port, timeout=5):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            print(f"✅ TCP reachable: {host}:{port}")
            return True
    except Exception as e:
        print(f"❌ TCP connection failed: {host}:{port}")
        print("   ", type(e).__name__, str(e))
        return False

qdrant_tcp_ok = test_tcp(SERVICE_NODE, QDRANT_PORT)
coordinator_tcp_ok = test_tcp(SERVICE_NODE, COORDINATOR_PORT)


✅ TCP reachable: gpub053.delta.ncsa.illinois.edu:6333
❌ TCP connection failed: gpub053.delta.ncsa.illinois.edu:8001
    ConnectionRefusedError [Errno 111] Connection refused


## 5. Qdrant HTTP API test

Calls `GET /collections`. This is read-only.


In [16]:
import requests

try:
    response = requests.get(f"{QDRANT_URL}/collections", timeout=10)
    print("HTTP status:", response.status_code)
    response.raise_for_status()
    data = response.json()
    print("✅ Qdrant HTTP API reachable")
    print(data)
except Exception as e:
    print("❌ Qdrant HTTP API test failed")
    print(type(e).__name__, str(e))
    raise


HTTP status: 200
✅ Qdrant HTTP API reachable
{'result': {'collections': [{'name': 'mirage_runtime_default_20260829_165430'}]}, 'status': 'ok', 'time': 5.881e-06}


## 6. Qdrant Python client test

This verifies the exact client style the preload worker will use.

If `qdrant-client` is not installed in this notebook environment, install it with:

```bash
pip install qdrant-client
```


In [17]:
try:
    from qdrant_client import QdrantClient

    client = QdrantClient(url=QDRANT_URL, timeout=10)
    collections = client.get_collections()
    print("✅ qdrant-client connection successful")
    print(collections)
except ImportError:
    print("❌ qdrant-client is not installed in this environment.")
    print("Run: pip install qdrant-client")
    raise
except Exception as e:
    print("❌ qdrant-client connection failed")
    print(type(e).__name__, str(e))
    raise


✅ qdrant-client connection successful
collections=[CollectionDescription(name='mirage_runtime_default_20260829_165430')]


## 7. Preload Coordinator health test

Calls `GET /health` on the coordinator. This is also read-only.


In [18]:
try:
    response = requests.get(f"{COORDINATOR_URL}/health", timeout=10)
    print("HTTP status:", response.status_code)
    response.raise_for_status()
    data = response.json()
    print("✅ Coordinator reachable")
    print(data)
except Exception as e:
    print("❌ Coordinator health test failed")
    print(type(e).__name__, str(e))
    raise


❌ Coordinator health test failed
ConnectionError HTTPConnectionPool(host='gpub053.delta.ncsa.illinois.edu', port=8001): Max retries exceeded with url: /health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7fdf04430c50>: Failed to establish a new connection: [Errno 111] Connection refused'))


ConnectionError: HTTPConnectionPool(host='gpub053.delta.ncsa.illinois.edu', port=8001): Max retries exceeded with url: /health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7fdf04430c50>: Failed to establish a new connection: [Errno 111] Connection refused'))

## 8. Combined result

If everything passes, the worker node can communicate with both shared services required by the concurrent preload architecture.


In [ ]:
results = {
    "worker_node": worker_fqdn,
    "service_node": SERVICE_NODE,
    "qdrant_url": QDRANT_URL,
    "coordinator_url": COORDINATOR_URL,
    "qdrant_tcp": bool(qdrant_tcp_ok),
    "coordinator_tcp": bool(coordinator_tcp_ok),
}

print("\n=== Connectivity Summary ===")
for k, v in results.items():
    print(f"{k:22}: {v}")

if qdrant_tcp_ok and coordinator_tcp_ok:
    print("\n✅ Basic cross-node connectivity is working.")
    print("If the HTTP and qdrant-client cells also passed, this node is ready to act as a preload worker.")
else:
    print("\n❌ At least one required port is not reachable from this worker node.")
    print("Check service-node hostname, server bind address, process status, and cluster/network policy.")
